# DermaMNIST + HDFS + Spark

The purpose of this notebook is to evaluate a Spark + HDFS data pipeline for the DermaMNIST dataset and compare it with the local baseline.

The dataset is stored in HDFS and accessed using Spark's binary-file reader.

After data is retrieved from HDFS, it is converted to PyTorch tensors and passed to the same CNN architecture used in the baseline experiment.

This notebook does not use distributed neural-network training. PyTorch training is performed by a single process. Distributed training
with PyTorch DDP is evaluated separately in the next experiment.

In [57]:
import io
import os
import sys
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, TensorDataset, DataLoader, WeightedRandomSampler

import pyspark

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PySpark:", pyspark.__version__)
print("PySpark location:", pyspark.__file__)

Python: 3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]
PyTorch: 2.14.1+cu130
PySpark: 4.1.2
PySpark location: /home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py


### 1. Reproducibility

We keep it identical to the baseline.

In [2]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

g = torch.Generator()
g.manual_seed(seed)

### 2. Prepare the dataset for HDFS + Spark

The original DermaMNIST dataset is provided as a single `.npz` archive containing all training, validation, and test images and labels.

In the first attempt, the entire `.npz` archive was read by Spark as a single binary object. This required the approximately 100 MB file to be serialized between the JVM and Python processes, resulting in excessive memory usage, and the first row of binary data frame never gets returned.

For the Spark experiment, the archive is therefore unpacked into individual image files organised by dataset split and class.

This representation allows Spark to access and process individual image files rather than transferring one large binary object.

The original `.npz` dataset remains unchanged and is used as the source for this preparation step.

In [11]:
# original local DermaMNIST dataset
npz_path = "/home/hduser/.medmnist/dermamnist_64.npz"

# local directory that will be uploaded to HDFS
output_dir = "/home/hduser/Desktop/CA1/dermamnist_spark"

os.makedirs(output_dir, exist_ok=True)

# load the original dataset
with np.load(npz_path) as data:
    datasets = {
        "train": (data["train_images"], data["train_labels"].flatten()),
        "val":   (data["val_images"], data["val_labels"].flatten()),
        "test":  (data["test_images"], data["test_labels"].flatten())
    }

# convert each image to PNG and organise by split/class
for split, (images, labels) in datasets.items():

    for class_id in np.unique(labels):
        class_dir = os.path.join(
            output_dir,
            split,
            str(class_id)
        )
        os.makedirs(class_dir, exist_ok=True)

    for i, (image, label) in enumerate(zip(images, labels)):
        image_path = os.path.join(
            output_dir,
            split,
            str(label),
            f"{i:05d}.png"
        )

        Image.fromarray(image).save(image_path)

print(f"Dataset prepared in: {output_dir}")

Dataset prepared in: /home/hduser/Desktop/CA1/dermamnist_spark


In [13]:
# check what we got
for split in ["train", "val", "test"]:
    count = len(list(
        Path(output_dir, split).rglob("*.png")
    ))
    print(f"{split}: {count} images")

train: 7007 images
val: 1003 images
test: 2005 images


#### Store the prepared dataset in HDFS

The prepared image dataset is now copied to HDFS. This upload is treated as experiment setup and is not included in the pipeline timing, just as the original DermaMNIST download was not included in the baseline timing.

### 3. Start Spark

Spark is used as the data-processing layer for the experiment. The pipeline timer starts immediately before the Spark session is created, so Spark startup overhead is included in the measured runtime.

In [18]:
# start timing the complete Spark + HDFS pipeline
pipeline_start = time.perf_counter()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DermaMNIST_Spark_HDFS")
    .master("local[*]")
    .getOrCreate()
)

spark_startup_time = time.perf_counter() - pipeline_start

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print(f"Spark startup time: {spark_startup_time:.2f} seconds")

Spark version: 4.1.2
Master: local[*]
Spark startup time: 0.01 seconds


In [19]:
# check the Hadoop filesystem we're using
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

print("Default filesystem:", hadoop_conf.get("fs.defaultFS"))

Default filesystem: hdfs://localhost:9000


### 4. Access the DermaMNIST dataset in HDFS

The prepared DermaMNIST image dataset has been uploaded to HDFS under:

    /ca1/dermamnist_spark/

The directory contains separate folders for the training, validation, and test sets, with images organised by class.

This structure allows Spark to treat the images as individual records instead of attempting to transfer the original ~100 MB `.npz` archive as one binary object.

In [20]:
hdfs_dataset_path = "hdfs:///ca1/dermamnist_spark"
print(hdfs_dataset_path)

hdfs:///ca1/dermamnist_spark


In [22]:
# read the image files with Spark
spark_read_start = time.perf_counter()

images_df = (spark.read.format("binaryFile").option("recursiveFileLookup", "true").load(hdfs_dataset_path))

# force Spark to actually perform the read
image_count = images_df.count()

spark_read_time = time.perf_counter() - spark_read_start

print(f"Images read by Spark: {image_count}")
print(f"Spark data-read time: {spark_read_time:.2f} seconds")

[Stage 5:===========================================>           (249 + 8) / 313]

Images read by Spark: 10015
Spark data-read time: 1.73 seconds


### 5. Inspect the dataset processed by Spark

Spark's `binaryFile` data source provides metadata for each image file, including its HDFS path, file size, modification time, and binary content.

For this experiment, the file path is also useful because it contains the dataset split and class label.

In [23]:
images_df.printSchema()

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)



In [24]:
images_df.select("path", "length").show(10, truncate=False)

+------------------------------------------------------------+------+
|path                                                        |length|
+------------------------------------------------------------+------+
|hdfs://localhost:9000/ca1/dermamnist_spark/train/5/04766.png|9812  |
|hdfs://localhost:9000/ca1/dermamnist_spark/train/5/01036.png|9732  |
|hdfs://localhost:9000/ca1/dermamnist_spark/train/5/05295.png|9598  |
|hdfs://localhost:9000/ca1/dermamnist_spark/val/5/00143.png  |9416  |
|hdfs://localhost:9000/ca1/dermamnist_spark/test/5/00965.png |9318  |
|hdfs://localhost:9000/ca1/dermamnist_spark/test/1/00376.png |9316  |
|hdfs://localhost:9000/ca1/dermamnist_spark/train/1/03488.png|9286  |
|hdfs://localhost:9000/ca1/dermamnist_spark/train/0/02810.png|9271  |
|hdfs://localhost:9000/ca1/dermamnist_spark/train/5/00243.png|9267  |
|hdfs://localhost:9000/ca1/dermamnist_spark/train/2/00110.png|9219  |
+------------------------------------------------------------+------+
only showing top 10 

### 6. Extract split and class using Spark

We can derive the split and class from the HDFS path.

In [25]:
from pyspark.sql.functions import regexp_extract

images_df = (
    images_df
    .withColumn(
        "split",
        regexp_extract("path", r"/(train|val|test)/", 1)
    )
    .withColumn(
        "class_id",
        regexp_extract("path", r"/(train|val|test)/([0-6])/", 2).cast("integer")
    )
)

images_df.select("split", "class_id", "path", "length").show(10, truncate=False)

+-----+--------+------------------------------------------------------------+------+
|split|class_id|path                                                        |length|
+-----+--------+------------------------------------------------------------+------+
|train|5       |hdfs://localhost:9000/ca1/dermamnist_spark/train/5/04766.png|9812  |
|train|5       |hdfs://localhost:9000/ca1/dermamnist_spark/train/5/01036.png|9732  |
|train|5       |hdfs://localhost:9000/ca1/dermamnist_spark/train/5/05295.png|9598  |
|val  |5       |hdfs://localhost:9000/ca1/dermamnist_spark/val/5/00143.png  |9416  |
|test |5       |hdfs://localhost:9000/ca1/dermamnist_spark/test/5/00965.png |9318  |
|test |1       |hdfs://localhost:9000/ca1/dermamnist_spark/test/1/00376.png |9316  |
|train|1       |hdfs://localhost:9000/ca1/dermamnist_spark/train/1/03488.png|9286  |
|train|0       |hdfs://localhost:9000/ca1/dermamnist_spark/train/0/02810.png|9271  |
|train|5       |hdfs://localhost:9000/ca1/dermamnist_spark/train/

### 7. Use Spark to verify the dataset distribution

Spark is used to calculate the number of images in each dataset split and class. This also verifies that the conversion to individual image files preserved the original DermaMNIST class distribution.

In [26]:
distribution = (images_df.groupBy("split", "class_id").count().orderBy("split", "class_id"))
distribution.show(30)

[Stage 10:==============================================>       (272 + 8) / 313]

+-----+--------+-----+
|split|class_id|count|
+-----+--------+-----+
| test|       0|   66|
| test|       1|  103|
| test|       2|  220|
| test|       3|   23|
| test|       4|  223|
| test|       5| 1341|
| test|       6|   29|
|train|       0|  228|
|train|       1|  359|
|train|       2|  769|
|train|       3|   80|
|train|       4|  779|
|train|       5| 4693|
|train|       6|   99|
|  val|       0|   33|
|  val|       1|   52|
|  val|       2|  110|
|  val|       3|   12|
|  val|       4|  111|
|  val|       5|  671|
|  val|       6|   14|
+-----+--------+-----+



### 8. Spark partitions

Spark divides the input data into partitions that can be processed in parallel. The number of partitions is inspected here to document the data-processing configuration used in the experiment.

In [27]:
print("Number of Spark partitions:", images_df.rdd.getNumPartitions())

Number of Spark partitions: 313


### 9. Transfer the processed dataset to Python

Spark has read the individual DermaMNIST image files from HDFS. The binary image content is now transferred to Python and converted
into NumPy arrays for PyTorch.

The original train/validation/test split and class labels are recovered from the file paths.

In [28]:
spark_to_python_start = time.perf_counter()

# select only the information needed by PyTorch
image_rows = (
    images_df
    .select("path", "content", "split", "class_id")
    .collect()
)

spark_to_python_time = time.perf_counter() - spark_to_python_start

print(f"Records transferred: {len(image_rows)}")
print(f"Spark → Python transfer: {spark_to_python_time:.2f} seconds")

Records transferred: 10015
Spark → Python transfer: 5.45 seconds


In [30]:
# convert the PNG bytes to NumPy
def decode_image(image_bytes):
    return np.array(Image.open(io.BytesIO(image_bytes)).convert("RGB"))

# test one image
image = decode_image(image_rows[0].content)

print("Shape:", image.shape)
print("Dtype:", image.dtype)
print("Range:", image.min(), image.max())

Shape: (64, 64, 3)
Dtype: uint8
Range: 32 222


In [50]:
# reconstruct the three datasets in their original image-index order
conversion_start = time.perf_counter()

def sort_by_original_index(rows):
    return sorted(
        rows,
        key=lambda row: int(row.path.rsplit("/", 1)[-1].replace(".png", ""))
    )

train_rows = sort_by_original_index([row for row in image_rows if row.split == "train"])
val_rows = sort_by_original_index([row for row in image_rows if row.split == "val"])
test_rows = sort_by_original_index([row for row in image_rows if row.split == "test"])

# convert to NumPy arrays
train_images = np.stack([decode_image(row.content) for row in train_rows])
train_labels = np.array([row.class_id for row in train_rows], dtype=np.int64)

val_images = np.stack([decode_image(row.content) for row in val_rows])
val_labels = np.array([row.class_id for row in val_rows], dtype=np.int64)

test_images = np.stack([decode_image(row.content) for row in test_rows])
test_labels = np.array([row.class_id for row in test_rows], dtype=np.int64)

conversion_time = time.perf_counter() - conversion_start

print("Train:", train_images.shape, train_labels.shape)
print("Val:  ", val_images.shape, val_labels.shape)
print("Test: ", test_images.shape, test_labels.shape)
print(f"Conversion time: {conversion_time:.2f} seconds")

Train: (7007, 64, 64, 3) (7007,)
Val:   (1003, 64, 64, 3) (1003,)
Test:  (2005, 64, 64, 3) (2005,)
Conversion time: 1.37 seconds


### 10. Prepare the data for PyTorch

The images have now been retrieved from HDFS through Spark and decoded into NumPy arrays.

From this point onward, we use the same preprocessing and training configuration as the local baseline. This ensures that differences in results are attributable to the Spark/HDFS data pipeline rather than changes to the CNN experiment.

The training set uses the same data augmentation and weighted oversampling as the baseline. Validation and test sets use only the
training-set normalization statistics.

In [53]:
# calculate normalization statistics from the Spark-retrieved training images
train_images_float = train_images.astype(np.float32) / 255.0

mean_spark = train_images_float.mean(axis=(0, 1, 2))
std_spark = train_images_float.std(axis=(0, 1, 2))

print("Spark/HDFS training mean:", mean_spark)
print("Spark/HDFS training std:", std_spark)

Spark/HDFS training mean: [0.5845583  0.58314306 0.5845583 ]
Spark/HDFS training std: [0.21778087 0.15745011 0.16998562]


### 11. Dataset class - PyTorch dataset adapter

The Spark pipeline produces the image data as NumPy arrays, whereas PyTorch's DataLoader expects a dataset implementing the standard Dataset interface. 

DermaMNISTArrayDataset class provides this interface by retrieving an image and label by index, converting the NumPy image to a PIL image for compatibility with torchvision transformations, applying the same preprocessing as the baseline, and returning the image and label as PyTorch tensors.

In [56]:
class DermaMNISTArrayDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # convert NumPy image back to PIL for torchvision transforms
        image = Image.fromarray(image)

        if self.transform is not None:
            image = self.transform(image)

        # keep label shape consistent with the baseline
        return image, torch.tensor([label], dtype=torch.long)

In [58]:
# same transforms as the baseline
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist())
])

transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist())
])

In [59]:
# create transformed datasets
train_dataset = DermaMNISTArrayDataset(train_images, train_labels, transform=transform_train)
val_dataset = DermaMNISTArrayDataset(val_images, val_labels, transform=transform_eval)
test_dataset = DermaMNISTArrayDataset(test_images, test_labels, transform=transform_eval)

In [60]:
# same weighted sampler
class_counts = np.bincount(train_labels)

class_sampling_weights = 1 / np.sqrt(class_counts)

sample_weights = class_sampling_weights[train_labels]

sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=g
)

In [61]:
# same DataLoaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

data_preparation_time = time.perf_counter() - pipeline_start

print(f"Data preparation time: {data_preparation_time:.4f}s")

Data preparation time: 5111.3410s


## 12. Model Initialization and Training

We use the same CNN architecture, loss, optimizer and learning rate as in the baseline. 

We use he same training and validation functions and training loop.

In [65]:
class CNN_deep(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [66]:
# initialise the model
model_start = time.perf_counter()

model = CNN_deep(num_classes=7)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

model_initialization_time = time.perf_counter() - model_start

print(f"Model initialization time: {model_initialization_time:.4f}s")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Model initialization time: 0.0098s
Number of parameters: 913,735


In [67]:
# training function
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        # remove the redundant dimension: (64, 1) → (64,)
        labels = labels.squeeze(1)

        optimizer.zero_grad()   # clear gradients from the previous batch

        outputs = model(images)   # perform the forward pass
        loss = criterion(outputs, labels)   # calculate the classification loss
        loss.backward()   # calculate gradients
        optimizer.step()   # update model parameters

        # CrossEntropyLoss() by default returns the average loss for the batch
        # we want the running_loss to represent the sum of the individual sample losses
        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [68]:
# validation function
def evaluate(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_predictions = []
    all_labels = []

    # we're telling PyTorch that this is evaluation rather than training, and that we don't need to calculate gradients
    with torch.no_grad():
        for images, labels in loader:
            labels = labels.squeeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            all_predictions.extend(predictions.numpy())
            all_labels.extend(labels.numpy())

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total
    epoch_macro_f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)

    return epoch_loss, epoch_accuracy, epoch_macro_f1

In [ ]:
num_epochs = 50

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": [],
    "epoch_time": []
}

best_macro_f1 = -1
best_epoch = 0
best_model_state = None

training_start = time.perf_counter()

for epoch in range(num_epochs):
    epoch_start = time.perf_counter()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)

    val_loss, val_acc, val_macro_f1 = evaluate(model, val_loader, criterion)

    epoch_time = time.perf_counter() - epoch_start

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)
    history["epoch_time"].append(epoch_time)

    # keep a copy of the model with the best validation macro-F1
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_epoch = epoch + 1
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:2d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f} | "
        f"Time: {epoch_time:.2f}s"
    )

training_time = time.perf_counter() - training_start

print(f"\nTraining time: {training_time:.2f}s")
print(f"Best validation macro-F1: {best_macro_f1:.4f}")
print(f"Best epoch: {best_epoch}")

### 13. Final Test Evaluation

We restore the model checkpoint with the highest validation macro-F1 before evaluating the test set.

The test set is evaluated only once at the end of the experiment and is not used for model selection.

In [ ]:
# restore the best model state
model.load_state_dict(best_model_state)

print(f"Using checkpoint from epoch {best_epoch}.")
print(f"Validation macro-F1: {best_macro_f1:.4f}")

In [ ]:
# time the test evaluation
test_start = time.perf_counter()

test_loss, test_acc, test_macro_f1 = evaluate(model, test_loader, criterion)

test_time = time.perf_counter() - test_start

print(f"Test evaluation time: {test_time:.2f}s")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test macro-F1: {test_macro_f1:.4f}")

In [ ]:
# collect predictions for per-class metrics
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        labels = labels.squeeze(1)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

print("Classification report:")
print(classification_report(all_labels, all_predictions, digits=4))

In [ ]:
# plot confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")
plt.title("Test Set Confusion Matrix")
plt.show()

[TBA]: Results conclusion

### 14. Timing Summary

The main stages of the HDFS + Spark pipeline are summarized below.

Training and test evaluation times include the time required to load and transform batches during iteration.

In [69]:
# summarise the measured pipeline stages
